# Diamond recursive meta-feedback: 10-minute WikiText-103 run

Full 10-minute GPU training on the recursive tokenizer-bus architecture:
bit/byte + subword/token/word + sentence, temporal meta slots, C(256,2)/C(256,3)
experts, per-layer feedback, and a meta NOW head.


In [ ]:
#@title GPU / TensorFlow
!nvidia-smi || true
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title Clone meta-feedback branch
%cd /kaggle/working
!rm -rf nanlm_experimental
!git clone --branch master --depth 1 https://github.com/shemshallah/nanlm_experimental.git
%cd /kaggle/working/nanlm_experimental
!git log --oneline -1


In [ ]:
#@title WikiText-103 download + article split
!if [ ! -s /kaggle/working/wiki.train.raw ]; then \
  wget -q --show-progress -O /kaggle/working/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!python diamond/colab/prepare_wikitext.py \
  --raw /kaggle/working/wiki.train.raw \
  --out /kaggle/working/corpus \
  --max-articles 4000 \
  --max-chars 20000000


In [ ]:
#@title 10-minute recursive meta-feedback training
!STATE_DIR=/kaggle/working/meta_10m \
 CORPUS_DIR=/kaggle/working/corpus \
 MODEL_SIZE=frontier BATCH=32 EVAL_BATCH=32 \
 EPOCHS=20 EXAMPLES=3000 CORPUS_CHARS=5000000 \
 TIME_BUDGET=10 TEST_SIZE=1024 \
 bash diamond/kaggle/train_kaggle.sh


In [ ]:
#@title Full metrics
!python diamond/colab/report_metrics.py /kaggle/working/meta_10m/diamond_metrics.json
